In [1]:
import pandas as pd
import re
from functools import reduce
from scipy.stats import pearsonr
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error 
from scipy.stats import zscore
import matplotlib.pyplot as plt 
from tensorflow.keras.models import load_model
import seaborn as sns
import math
import random

In [2]:
tf.keras.backend.clear_session()

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

### Code to run FCNN to compute CV

In [3]:
tcga_features = pd.read_csv("/data/kumarr17/common_features_ver3.txt", header=None)
cyt_features = pd.read_csv("/data/kumarr17/common_cyt_nichenet_gulden_tcga.txt", header=None)
tcga_features_list=list(tcga_features[0])
cyt_features_list=list(cyt_features[0])

In [4]:
df_tcga=pd.read_csv("/data/kumarr17/merged_sample_mtx/TCGA_Merged_mRNA_Expression.tsv", index_col=0)

In [5]:
df_tcga_features=df_tcga[tcga_features_list]
df_tcga_cyt_col=df_tcga[cyt_features_list]
filtered_model_feature_list=tcga_features_list

In [10]:
import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# -----------------------------
# speed settings
# -----------------------------
tf.keras.backend.clear_session()

cyt_tmp = cyt_features_list#cyt_of_interest2
corr_new_tmp = []

# features
X_raw = df_tcga_features.values.astype("float32")

# optional: row-wise z-score, faster than apply(zscore, axis=1)
X_raw = (X_raw - X_raw.mean(axis=1, keepdims=True)) / (
    X_raw.std(axis=1, keepdims=True) + 1e-8
)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

def build_fcnn(input_dim):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(128, activation="relu"),
        layers.Dense(64, activation="relu"),
        layers.Dense(32, activation="relu"),
        layers.Dense(1)
    ])
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="mean_squared_error",
        metrics=["mae"]
    )
    return model

early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

for cytokine_to_check in cyt_tmp:
    
    print(f"\n==============================")
    print(f"Cytokine: {cytokine_to_check}")
    print(f"==============================")
    
    y = df_tcga_cyt_col[[cytokine_to_check]].values.astype("float32")
    
    corr_lt = []
    
    for fold, (train_index, val_index) in enumerate(kf.split(X_raw)):
        print(f"Fold {fold+1}/5")
        
        X_train_raw = X_raw[train_index]
        X_val_raw   = X_raw[val_index]
        y_train     = y[train_index]
        y_val       = y[val_index]
        
        # scale using training fold only
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train_raw).astype("float32")
        X_val   = scaler.transform(X_val_raw).astype("float32")
        
        tf.keras.backend.clear_session()
        model = build_fcnn(X_train.shape[1])
        
        model.fit(
            X_train,
            y_train,
            validation_data=(X_val, y_val),
            epochs=50,
            batch_size=128,      # larger batch = faster
            verbose=0,
            callbacks=[early_stop]
        )
        
        y_pred = model.predict(X_val, verbose=0).ravel()
        y_true = y_val.ravel()
        
        corr, p_value = pearsonr(y_true, y_pred)
        corr_lt.append(corr)
    
    mean_corr = np.mean(corr_lt)
    corr_new_tmp.append(mean_corr)
    
    print("Mean CV Pearson:", mean_corr)

CV_df = pd.DataFrame(
    {"CV": corr_new_tmp},
    index=cyt_tmp
)
CV_df.to_csv("/data/kumarr17/fixed_feature_cyt_result/FCNN_ridge_xgboost_cv/FCNN_cv.csv")


Cytokine: INHA
Fold 1/5


2026-09-15 10:20:50.700893: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcuda.so.1'; dlerror: libcuda.so.1: cannot open shared object file: No such file or directory
2026-09-15 10:20:50.700927: W tensorflow/stream_executor/cuda/cuda_driver.cc:269] failed call to cuInit: UNKNOWN ERROR (303)
2026-09-15 10:20:50.700985: I tensorflow/stream_executor/cuda/cuda_diagnostics.cc:156] kernel driver does not appear to be running on this host (cn0013): /proc/driver/nvidia/version does not exist
2026-09-15 10:20:50.701248: I tensorflow/core/platform/cpu_feature_guard.cc:151] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 AVX512F FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


Fold 2/5
Fold 3/5
Fold 4/5
Fold 5/5
Mean CV Pearson: 0.7722758940214846

Cytokine: FGF7
Fold 1/5
Fold 2/5
Fold 3/5
Fold 4/5
Fold 5/5
Mean CV Pearson: 0.8106003947637195

Cytokine: BMP6
Fold 1/5
Fold 2/5
Fold 3/5
Fold 4/5
Fold 5/5
Mean CV Pearson: 0.59607567118479

Cytokine: IL25
Fold 1/5
Fold 2/5
Fold 3/5
Fold 4/5
Fold 5/5
Mean CV Pearson: 0.23790501009163298

Cytokine: CCL21
Fold 1/5
Fold 2/5
Fold 3/5
Fold 4/5
Fold 5/5
Mean CV Pearson: 0.6898322552385558

Cytokine: LIPH
Fold 1/5
Fold 2/5
Fold 3/5
Fold 4/5
Fold 5/5
Mean CV Pearson: 0.9117977036367393

Cytokine: SFTPA1
Fold 1/5
Fold 2/5
Fold 3/5
Fold 4/5
Fold 5/5
Mean CV Pearson: 0.8348990945610142

Cytokine: FGF23
Fold 1/5
Fold 2/5
Fold 3/5
Fold 4/5
Fold 5/5
Mean CV Pearson: 0.3501096270223164

Cytokine: ADCYAP1
Fold 1/5
Fold 2/5
Fold 3/5
Fold 4/5
Fold 5/5
Mean CV Pearson: 0.6251250888751257

Cytokine: MIA
Fold 1/5
Fold 2/5
Fold 3/5
Fold 4/5
Fold 5/5
Mean CV Pearson: 0.8540794715521066

Cytokine: VCAM1
Fold 1/5
Fold 2/5
Fold 3/5
Fold 4

#### Code for running ridge regression to compute CV

In [6]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [7]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import Ridge
from scipy.stats import pearsonr


# Features: samples × genes
X = df_tcga_features

# Targets: samples × cytokines
Y = df_tcga_cyt_col


# Same CV as FCNN
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


ridge_results = []
ridge_prediction_df = pd.DataFrame(
    index=X.index,
    columns=Y.columns,
    dtype=float
)


for cytokine in Y.columns:

    print("Running Ridge:", cytokine)

    y = Y[cytokine]

    # Keep samples with available target
    valid = y.notna()

    X_temp = X.loc[valid]
    y_temp = y.loc[valid]


    # IMPORTANT:
    # scaler is fitted ONLY on training fold
    model = make_pipeline(
        StandardScaler(),
        Ridge(
            alpha=1.0
        )
    )


    # Held-out prediction for every sample
    y_pred = cross_val_predict(
        model,
        X_temp,
        y_temp,
        cv=kf,
        n_jobs=-1
    )


    # CV Pearson correlation
    r, p = pearsonr(
        y_temp.values,
        y_pred
    )


    ridge_results.append({
        'cytokine': cytokine,
        'Ridge_CV': r,
        'pvalue': p
    })


    ridge_prediction_df.loc[
        y_temp.index,
        cytokine
    ] = y_pred


ridge_cv_df = pd.DataFrame(ridge_results)

ridge_cv_df.head()
ridge_cv_df.to_csv("/data/kumarr17/fixed_feature_cyt_result/FCNN_ridge_xgboost_cv/ridge_cv.csv")

Running Ridge: INHA
Running Ridge: FGF7
Running Ridge: BMP6
Running Ridge: IL25
Running Ridge: CCL21
Running Ridge: LIPH
Running Ridge: SFTPA1
Running Ridge: FGF23
Running Ridge: ADCYAP1
Running Ridge: MIA
Running Ridge: VCAM1
Running Ridge: BMP15
Running Ridge: ARF1
Running Ridge: ANGPTL4
Running Ridge: PODXL2
Running Ridge: IL18
Running Ridge: FGF21
Running Ridge: COL4A5
Running Ridge: FAT4
Running Ridge: MADCAM1
Running Ridge: RGMA
Running Ridge: PDGFC
Running Ridge: PLAT
Running Ridge: ADAM28
Running Ridge: IL24
Running Ridge: DEFB1
Running Ridge: FN1
Running Ridge: PNOC
Running Ridge: CAMP
Running Ridge: CSHL1
Running Ridge: OLFM2
Running Ridge: A2M
Running Ridge: CEL
Running Ridge: GRP
Running Ridge: MUC7
Running Ridge: MLN
Running Ridge: IL1F10
Running Ridge: CCL20
Running Ridge: PMCH
Running Ridge: SPON2
Running Ridge: CXCL11
Running Ridge: GDF6
Running Ridge: VWF
Running Ridge: RSPO3
Running Ridge: SERPINE2
Running Ridge: ZG16B
Running Ridge: NPPB
Running Ridge: BTLA
Running R

#### Code for running XGboost to compute CV

In [6]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [7]:
import numpy as np
import pandas as pd

from xgboost import XGBRegressor
from sklearn.model_selection import KFold
from scipy.stats import pearsonr


X = df_tcga_features.astype(np.float32)
Y = df_tcga_cyt_col.astype(np.float32)

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

results = []
prediction_df = pd.DataFrame(
    index=X.index,
    columns=Y.columns,
    dtype=np.float32
)

# Precompute folds ONCE
folds = list(kf.split(X))

for cytokine in Y.columns:

    print("Running:", cytokine)

    y = Y[cytokine]

    valid = y.notna()

    X_temp = X.loc[valid]
    y_temp = y.loc[valid]

    # If there are no missing targets, the same folds can be reused directly
    # Otherwise regenerate folds for X_temp
    if valid.all():
        current_folds = folds
    else:
        current_folds = list(kf.split(X_temp))

    y_pred = np.empty(len(y_temp), dtype=np.float32)

    for train_idx, test_idx in current_folds:

        model = XGBRegressor(
            objective="reg:squarederror",

            # Much faster baseline
            n_estimators=200,
            learning_rate=0.08,
            max_depth=4,

            subsample=0.8,
            colsample_bytree=0.5,

            reg_lambda=1,

            # Very important for speed
            tree_method="hist",

            random_state=42,

            # Let XGBoost parallelize
            n_jobs=-1
        )

        model.fit(
            X_temp.iloc[train_idx],
            y_temp.iloc[train_idx],
            verbose=False
        )

        y_pred[test_idx] = model.predict(
            X_temp.iloc[test_idx]
        )

    r, p = pearsonr(
        y_temp.to_numpy(),
        y_pred
    )

    results.append({
        "cytokine": cytokine,
        "XGBoost_CV": r,
        "pvalue": p
    })

    prediction_df.loc[y_temp.index, cytokine] = y_pred

    # -----------------------------------------
    # SAVE AFTER EVERY CYTOKINE
    # -----------------------------------------

    xgboost_cv_df = pd.DataFrame(results)

    xgboost_cv_df.to_csv(
        "/data/kumarr17/xgboost_result/xgboost_cv_results.csv",
        index=False
    )

    prediction_df.to_csv(
        "/data/kumarr17/xgboost_result/xgboost_cv_predictions.csv"
    )

    print(
        f"Finished {cytokine}: "
        f"r={r:.4f}, p={p:.3e} | Results saved"
    )


xgboost_cv_df = pd.DataFrame(results)
xgboost_cv_df.to_csv("/data/kumarr17/fixed_feature_cyt_result/xgboost_cv.csv")

Running: INHA
Finished INHA: r=0.7231, p=0.000e+00 | Results saved
Running: FGF7
Finished FGF7: r=0.7713, p=0.000e+00 | Results saved
Running: BMP6
Finished BMP6: r=0.5227, p=0.000e+00 | Results saved
Running: IL25
Finished IL25: r=0.0026, p=7.996e-01 | Results saved
Running: CCL21
Finished CCL21: r=0.7868, p=0.000e+00 | Results saved
Running: LIPH
Finished LIPH: r=0.8696, p=0.000e+00 | Results saved
Running: SFTPA1
Finished SFTPA1: r=0.7751, p=0.000e+00 | Results saved
Running: FGF23
Finished FGF23: r=0.1764, p=3.465e-67 | Results saved
Running: ADCYAP1
Finished ADCYAP1: r=0.4421, p=0.000e+00 | Results saved
Running: MIA


KeyboardInterrupt: 